In [1]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import optuna
import warnings
import logging

warnings.filterwarnings('ignore', category=FutureWarning)
optuna.logging.set_verbosity(logging.WARNING)

# ==========================================
# 1. LOAD AND PREPARE HOURLY DATA
# ==========================================
# I'm loading all my preprocessed data files from the same pipeline every model uses
print("Loading and filtering data...")

data_dir = '../preprocesing_data/processed_csv'
sales_path = os.path.join(data_dir, 'sales_data_preprocessed.csv')
weather_path = os.path.join(data_dir, 'weather_data_hourly.csv')
holidays_path = os.path.join(data_dir, 'holidays_data_preprocessed.csv')
events_path = os.path.join(data_dir, 'aberdeen_events_master_timeline.csv')

# --- 1.1 Sales (Hourly) ---
sales_df = pd.read_csv(sales_path)
sales_df['Date'] = pd.to_datetime(sales_df['Date'])

# Filter to restaurant opening hours only: 08:00 to 16:00 (9 hours, store closes at 17:00)
sales_df = sales_df[(sales_df['Date'].dt.hour >= 8) & (sales_df['Date'].dt.hour < 17)]

# Aggregate to hourly (in case of sub-hourly rows)
hourly_sales = sales_df.groupby('Date').sum(numeric_only=True).reset_index()

# Identify product columns (everything except Date)
product_cols = [c for c in hourly_sales.columns if c != 'Date']
print(f"Products found: {product_cols}")

# MELT: wide -> long (one row per product per hour)
df_long = pd.melt(hourly_sales, id_vars=['Date'], value_vars=product_cols,
                  var_name='Product_Name', value_name='Sales')
df_long['Product_Name'] = df_long['Product_Name'].astype('category')
df_long['Sales'] = df_long['Sales'].clip(lower=0)

# Only keep my 74 target products

# ==========================================
# PRODUCTS TO FORECAST
# ==========================================
# These are the 74 specific menu items I want to predict
PRODUCTS_TO_FORECAST = [
 'Avo & Hal Muffin',
 'Avo, Egg & Bacon',
 'Avo, Feta & Tom',
 'Avocado on Toast',
 'Bacon',
 'Bacon Bap',
 'Bacon Egg Brioch',
 'Bacon Waffle',
 'Baked Beans',
 'Baked Beans JP',
 'Bean Soldiers',
 'Big Breakfast',
 'Black Pudding',
 'Breakfast Hash',
 'Breakfast Muffin',
 'Breakfast Wrap',
 'Buttd Mushrooms',
 'Cheese & Bean JP',
 'Cheese JP',
 'Chick Flatbread',
 'Chicken Club',
 'Chilli Carne JP',
 'Egg Bacon Brioch',
 'Egg Bap',
 'Extra Beans',
 'F.Eggs on Toast',
 'Festive Stack',
 'Fried Egg',
 'Hash Brown',
 'Hash Brown Bites',
 'Little Avo Toast',
 'Little Bean Toas',
 'Little Egg Toast',
 'Ltle Bfast Bacon',
 'Ltle Bfast Saus',
 'Mini Hash Browns',
 'P.Eggs on Toast',
 'Poached Egg',
 'Posh Beans',
 'Roll & Butter ',
 'S.Eggs on Toast',
 'Sausage',
 'Sausage Bap',
 'Scrambled Egg',
 'Streaky Bacon',
 'Tattie Scone',
 'The Breakfast',
 'Toasted Teacake',
 'Tuna JP',
 'Tuna Mayo Mix',
 'Tuna Melt Panini',
 'Tuna Panini',
 'Tuna Toastie',
 'Veg Sausage Bap',
 'Vegan Breakfast',
 'Vegan Sausage',
 'Veggie Bap',
 'Veggie Breakfast',
 'Bakery',
 'White Toast Bread',
 'Brown Toast Bread',
 'Porridge',
 'Sourdough Toast Bread',
 'Multiseed Toast Bread',
]
df_long = df_long[df_long['Product_Name'].isin(PRODUCTS_TO_FORECAST)].reset_index(drop=True)

# --- 1.2 Weather (Hourly) ---
weather_df = pd.read_csv(weather_path)
weather_df['Date'] = pd.to_datetime(weather_df['Date'])

# Create weather flags
if 'weather_code' in weather_df.columns:
    weather_df['is_clear'] = (weather_df['weather_code'] == 0).astype(int)
    weather_df['is_cloudy'] = weather_df['weather_code'].isin([1, 2, 3, 45, 48]).astype(int)
    weather_df['is_rain'] = weather_df['weather_code'].isin(
        [51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82, 95, 96, 99]).astype(int)
    weather_df['is_snow'] = weather_df['weather_code'].isin([71, 73, 75, 77, 85, 86]).astype(int)
    weather_df = weather_df.drop(columns=['weather_code'])

# Weather is already hourly — merge directly
hourly_weather = weather_df.copy()

# --- 1.3 Holidays (Daily -> broadcast to hourly) ---
holidays_df = pd.read_csv(holidays_path)
holidays_df['Date'] = pd.to_datetime(holidays_df['Date'])
daily_holidays = holidays_df.groupby(holidays_df['Date'].dt.normalize()).max(numeric_only=True).reset_index()
daily_holidays = daily_holidays.rename(columns={'Date': 'Date_Only'})

# Holiday proximity features
daily_holidays['is_holiday_lag_1'] = daily_holidays['is_holiday'].shift(1).fillna(0)
daily_holidays['is_holiday_lead_1'] = daily_holidays['is_holiday'].shift(-1).fillna(0)

# --- 1.4 Events (Daily -> broadcast to hourly) ---
events_df = pd.read_csv(events_path)
events_df['Date'] = pd.to_datetime(events_df['Date'])
events_df['Date_Only'] = events_df['Date'].dt.normalize()
daily_events = events_df.groupby('Date_Only').max(numeric_only=True).reset_index()

print(f"Hourly sales rows: {len(df_long)}")
print(f"Date range: {df_long['Date'].min()} to {df_long['Date'].max()}")


/home/alex/miniconda3/envs/jupyterproject-project/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading and filtering data...
Products found: ['Almd & Aprct Bar', 'Apl & Blk Smthie', 'Avo & Hal Muffin', 'Avo, Egg & Bacon', 'Avo, Feta & Tom', 'Avocado on Toast', 'BD Curry Roll', 'BLT', 'Bacon', 'Bacon Bap', 'Bacon Egg Brioch', 'Bacon Waffle', 'Baked Beans', 'Baked Beans JP', 'Banana Bread', 'Bean Soldiers', 'Beef Burger', 'Beef Hors Foca', 'Beyond Burger', 'Big Breakfast', 'Black Pudding', 'Blk Forest Syrup', 'Bre Cran Toastie', 'Breakfast Bap', 'Breakfast Hash', 'Breakfast Muffin', 'Breakfast Wrap', 'Brie Bacon Crois', 'Brie Cranb Tstie', 'Brunch Burger', 'Buttd Mushrooms', 'Butter', 'Cheese & Bean JP', 'Cheese JP', 'Cheese Pizza', 'Cheese Sand PnM', 'Cheese Sandwich', 'Cheese Scone', 'Chick Chzo Tstie', 'Chick Flatbread', 'Chick Rice Bowl', 'Chicken Breast', 'Chicken Burger', 'Chicken Club', 'Chicken Goujons', 'Chicken Strips', 'Chicken Waffles', 'Chicken Wrap', 'Chickn Rice Bowl', 'Chilli Carne JP', 'Chips', 'Chorizo', 'Chorizo & Eggs', 'Chse Onion Tstie', 'Chse Tom Toastie', '

In [2]:
# ==========================================
# 2. FEATURE ENGINEERING (HOURLY)
# ==========================================
print("Engineering hourly features...")

# --- Cyclical time features ---
# Hourly: cyclical encoding for the 9-hour window (08:00-16:00)
df_long['hour_of_day'] = df_long['Date'].dt.hour
df_long['hour_sin'] = np.sin(2 * np.pi * (df_long['hour_of_day'] - 8) / 9)
df_long['hour_cos'] = np.cos(2 * np.pi * (df_long['hour_of_day'] - 8) / 9)

# Daily: cyclical day of week
df_long['day_of_week'] = df_long['Date'].dt.dayofweek
df_long['day_sin'] = np.sin(2 * np.pi * df_long['day_of_week'] / 7)
df_long['day_cos'] = np.cos(2 * np.pi * df_long['day_of_week'] / 7)

# Monthly: cyclical month
df_long['month'] = df_long['Date'].dt.month
df_long['month_sin'] = np.sin(2 * np.pi * (df_long['month'] - 1) / 12)
df_long['month_cos'] = np.cos(2 * np.pi * (df_long['month'] - 1) / 12)

# Binary features
df_long['Is_Weekend'] = df_long['day_of_week'].isin([5, 6]).astype(int)
df_long['day_of_month'] = df_long['Date'].dt.day
df_long['week_of_year'] = df_long['Date'].dt.isocalendar().week.astype(int)

# Hourly position flags (peak hours vs early/late)
# Lunch rush typically 11:00-14:00, morning rush 08:00-10:00
df_long['is_morning_rush'] = df_long['hour_of_day'].isin([8, 9, 10]).astype(int)
df_long['is_lunch_rush'] = df_long['hour_of_day'].isin([11, 12, 13, 14]).astype(int)
df_long['is_afternoon'] = df_long['hour_of_day'].isin([15, 16]).astype(int)

# --- Merge date key for daily features ---
df_long['Date_Only'] = df_long['Date'].dt.normalize()

# Merge hourly weather
df = df_long.merge(hourly_weather, on='Date', how='left')

# Merge daily holidays
df = df.merge(daily_holidays, on='Date_Only', how='left')

# Merge daily events
event_feature_cols = [c for c in daily_events.columns if c != 'Date_Only']
df = df.merge(daily_events[['Date_Only'] + event_feature_cols], on='Date_Only', how='left')

# Drop temporary columns
for col in ['Date_Only', 'hour_of_day', 'day_of_week', 'month']:
    if col in df.columns:
        df = df.drop(columns=[col])

# Drop any stray object columns
for col in ['date', 'Time']:
    if col in df.columns:
        df = df.drop(columns=[col])

# Fill NaN for non-categorical columns
categorical_cols = df.select_dtypes(include=['category']).columns
non_categorical_cols = df.columns.difference(categorical_cols)
df[non_categorical_cols] = df[non_categorical_cols].fillna(0)

# Clip flag columns
flag_cols = [c for c in df.columns if c.startswith('is_')]
if flag_cols:
    df[flag_cols] = df[flag_cols].clip(upper=1)

print(f"Shape after merges: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Engineering hourly features...
Shape after merges: (841536, 37)
Columns: ['Date', 'Product_Name', 'Sales', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'Is_Weekend', 'day_of_month', 'week_of_year', 'is_morning_rush', 'is_lunch_rush', 'is_afternoon', 'apparent_temperature', 'precipitation', 'snowfall', 'snow_depth', 'relative_humidity_2m', 'cloud_cover', 'visibility', 'wind_speed_10m', 'wind_gusts_10m', 'is_clear', 'is_cloudy', 'is_rain', 'is_snow', 'is_holiday', 'holiday_importance', 'Days_Until_Next_Holiday', 'is_holiday_lag_1', 'is_holiday_lead_1', 'is_festival', 'festival_importance', 'is_match_day', 'match_importance']


In [3]:
# ==========================================
# 3. HOURLY LAG & ROLLING FEATURES
# ==========================================
print("Creating hourly lag and rolling features...")

# Sort by product and date for correct lag computation
df = df.sort_values(['Product_Name', 'Date']).reset_index(drop=True)

# HOURS_PER_DAY: 9 hours of operation (08:00-16:00, store closes at 17:00)
HOURS_PER_DAY = 9

# Hourly lags: 1 hour ago, 2 hours ago, same hour yesterday (9 steps),
# same hour 1 week ago (63 steps), same hour 2 weeks ago (126 steps)
HOURLY_LAGS = {
    1: 'sales_1h_ago',
    2: 'sales_2h_ago',
    HOURS_PER_DAY: 'sales_same_hour_yesterday',
    HOURS_PER_DAY * 7: 'sales_same_hour_last_week',
    HOURS_PER_DAY * 14: 'sales_same_hour_2weeks_ago'
}

for lag_steps, col_name in HOURLY_LAGS.items():
    df[col_name] = df.groupby('Product_Name', observed=False)['Sales'].shift(lag_steps)

# Rolling features on 1-hour lag (within each product)
ROLLING_WINDOWS_HOURS = [HOURS_PER_DAY, HOURS_PER_DAY * 3, HOURS_PER_DAY * 7]  # 1 day, 3 days, 1 week
ROLLING_WINDOW_NAMES = ['1d', '3d', '7d']

for w, name in zip(ROLLING_WINDOWS_HOURS, ROLLING_WINDOW_NAMES):
    df[f'rolling_{name}_avg'] = df.groupby('Product_Name', observed=False)['sales_1h_ago'].transform(
        lambda x: x.rolling(w, min_periods=1).mean()
    )
    df[f'rolling_{name}_std'] = df.groupby('Product_Name', observed=False)['sales_1h_ago'].transform(
        lambda x: x.rolling(w, min_periods=1).std()
    ).fillna(0)

# Sales momentum (hourly)
df['sales_momentum_hourly'] = df['sales_1h_ago'] - df['sales_same_hour_yesterday']

# Expanding mean per product (no leakage — uses shift)
df['expanding_mean'] = df.groupby('Product_Name', observed=False)['sales_1h_ago'].transform(
    lambda x: x.expanding(min_periods=1).mean()
)

# Ratio: last hour vs rolling daily average
df['ratio_1h_vs_1d_avg'] = df['sales_1h_ago'] / (df['rolling_1d_avg'] + 1e-8)

# Drop temp columns and NaNs
df = df.dropna().reset_index(drop=True)

# Define feature columns (everything except Date and Sales)
feature_cols = [c for c in df.columns if c not in ['Date', 'Sales']]

print(f"Final shape: {df.shape}")
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Products: {df['Product_Name'].cat.categories.tolist()}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")

Creating hourly lag and rolling features...
Final shape: (833472, 51)
Features (49): ['Product_Name', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'Is_Weekend', 'day_of_month', 'week_of_year', 'is_morning_rush', 'is_lunch_rush', 'is_afternoon', 'apparent_temperature', 'precipitation', 'snowfall', 'snow_depth', 'relative_humidity_2m', 'cloud_cover', 'visibility', 'wind_speed_10m', 'wind_gusts_10m', 'is_clear', 'is_cloudy', 'is_rain', 'is_snow', 'is_holiday', 'holiday_importance', 'Days_Until_Next_Holiday', 'is_holiday_lag_1', 'is_holiday_lead_1', 'is_festival', 'festival_importance', 'is_match_day', 'match_importance', 'sales_1h_ago', 'sales_2h_ago', 'sales_same_hour_yesterday', 'sales_same_hour_last_week', 'sales_same_hour_2weeks_ago', 'rolling_1d_avg', 'rolling_1d_std', 'rolling_3d_avg', 'rolling_3d_std', 'rolling_7d_avg', 'rolling_7d_std', 'sales_momentum_hourly', 'expanding_mean', 'ratio_1h_vs_1d_avg']
Products: ['Almd & Aprct Bar', 'Apl & Blk Smthie', 'Av

In [4]:
# ==========================================
# 4. TRAIN / VALIDATION / TEST SPLIT
# ==========================================
print("Splitting data...")

# My train/test split dates — I'm holding out November as a blind test
train_end = pd.to_datetime('2025-10-01')
val_end = pd.to_datetime('2025-11-01')
test_end = pd.to_datetime('2025-11-30 23:59:59')

train_df = df[df['Date'] <= train_end].copy()
val_df = df[(df['Date'] > train_end) & (df['Date'] <= val_end)].copy()
test_df = df[(df['Date'] > val_end) & (df['Date'] <= test_end)].copy().reset_index(drop=True)

X_train, y_train = train_df[feature_cols], train_df['Sales']
X_val, y_val = val_df[feature_cols], val_df['Sales']

print(f"Train: {len(train_df)} rows ({train_df['Date'].min()} to {train_df['Date'].max()})")
print(f"Val:   {len(val_df)} rows ({val_df['Date'].min()} to {val_df['Date'].max()})")
print(f"Test:  {len(test_df)} rows ({test_df['Date'].min()} to {test_df['Date'].max()})")

Splitting data...
Train: 780480 rows (2022-01-15 08:00:00 to 2025-09-30 16:00:00)
Val:   17856 rows (2025-10-01 08:00:00 to 2025-10-31 16:00:00)
Test:  17280 rows (2025-11-01 08:00:00 to 2025-11-30 16:00:00)


In [5]:
# ==========================================
# 5. OPTUNA HYPERPARAMETER TUNING
# ==========================================
print("Running Optuna Hyperparameter Tuning (75 trials)...")


def objective(trial):
    param = {
        "n_estimators": 2000,
        "early_stopping_rounds": 50,
        "learning_rate": trial.suggest_float("learning_rate", 3e-3, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 2, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 0.9),
        "gamma": trial.suggest_float("gamma", 1e-4, 2.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 10.0, log=True),
        "enable_categorical": True,
        "tree_method": "hist"
    }

    model = xgb.XGBRegressor(**param)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    return rmse


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
best_params = study.best_params
best_params.update({
    "n_estimators": 2000,
    "early_stopping_rounds": 30,
    "enable_categorical": True,
    "tree_method": "hist"
})

print(f"\nBest Validation RMSE: {study.best_value:.4f}")
print(f"Best params:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

Running Optuna Hyperparameter Tuning (75 trials)...

Best Validation RMSE: 0.8596
Best params:
  learning_rate: 0.01124798846924921
  max_depth: 9
  min_child_weight: 3
  subsample: 0.9124379879428419
  colsample_bytree: 0.528724290064385
  gamma: 0.7554736023395866
  reg_lambda: 0.31942283641188407
  reg_alpha: 0.13500175390392047
  n_estimators: 2000
  early_stopping_rounds: 30
  enable_categorical: True
  tree_method: hist


In [6]:
# ==========================================
# 6. TRAIN FINAL MODEL
# ==========================================
print("Training final global model with best parameters...")

model = xgb.XGBRegressor(**best_params)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

print("Model trained.")

Training final global model with best parameters...
Model trained.


In [7]:
# ==========================================
# 7. RECURSIVE HOURLY FORECAST — ALL PRODUCTS
# ==========================================
print("Running Recursive Hourly Forecast for ALL Products...")

HOURS_PER_DAY = 9

# Hourly lag map: lag_steps -> column name
HOURLY_LAG_MAP = {
    1: 'sales_1h_ago',
    2: 'sales_2h_ago',
    HOURS_PER_DAY: 'sales_same_hour_yesterday',
    HOURS_PER_DAY * 7: 'sales_same_hour_last_week',
    HOURS_PER_DAY * 14: 'sales_same_hour_2weeks_ago'
}

ROLLING_WINDOWS_HOURS = [HOURS_PER_DAY, HOURS_PER_DAY * 3, HOURS_PER_DAY * 7]
ROLLING_WINDOW_NAMES = ['1d', '3d', '7d']

all_products = test_df['Product_Name'].cat.categories.tolist()
test_hours = sorted(test_df['Date'].unique())

# Prepare prediction storage
test_df['XGB_Prediction'] = 0.0
test_df['original_sales_1h_ago'] = test_df['sales_1h_ago'].copy()

# Build lookup: (product, datetime) -> row index
idx_lookup = {}
for idx, row in test_df.iterrows():
    idx_lookup[(row['Product_Name'], row['Date'])] = idx

# Process hour by hour, all products at once
for hour_i, current_hour in enumerate(test_hours):
    if hour_i % (HOURS_PER_DAY * 5) == 0:
        print(f"  Processing hour {hour_i+1}/{len(test_hours)}: {current_hour}")

    # Get all product rows for this hour
    hour_mask = test_df['Date'] == current_hour
    hour_indices = test_df.index[hour_mask].tolist()

    if len(hour_indices) == 0:
        continue

    # Predict all products for this hour
    X_hour = test_df.loc[hour_indices, feature_cols]
    preds = model.predict(X_hour)
    # Clip negative predictions to zero — we can't sell negative items
    preds = np.clip(preds, 0, None)

    # Store predictions
    test_df.loc[hour_indices, 'XGB_Prediction'] = preds

    # Feed predictions forward into future hourly lag columns
    for row_idx, pred_val in zip(hour_indices, preds):
        product = test_df.at[row_idx, 'Product_Name']

        for lag_steps, lag_col in HOURLY_LAG_MAP.items():
            # Future datetime = current_hour + lag_steps hours of operation
            # We need to find the actual datetime that is lag_steps operational hours ahead
            future_key = (product, None)  # placeholder

            # Find the position of current hour in the product's test timeline
            # and jump lag_steps forward
            product_mask = test_df['Product_Name'] == product
            product_hours = test_df.loc[product_mask, 'Date'].values
            current_pos = np.searchsorted(product_hours, np.datetime64(current_hour))
            future_pos = current_pos + lag_steps

            if future_pos < len(product_hours):
                future_datetime = pd.Timestamp(product_hours[future_pos])
                future_key = (product, future_datetime)
                if future_key in idx_lookup:
                    test_df.at[idx_lookup[future_key], lag_col] = pred_val

    # Rebuild derived features for the NEXT hour
    if hour_i + 1 < len(test_hours):
        next_hour = test_hours[hour_i + 1]
        next_mask = test_df['Date'] == next_hour
        next_indices = test_df.index[next_mask].tolist()

        for next_idx in next_indices:
            product = test_df.at[next_idx, 'Product_Name']

            # Get this product's data in test
            product_mask = test_df['Product_Name'] == product
            product_df = test_df.loc[product_mask].sort_values('Date')
            product_dates = product_df['Date'].values
            pos = np.searchsorted(product_dates, np.datetime64(next_hour))

            # Rebuild rolling stats from sales_1h_ago history
            lag1_history = product_df['sales_1h_ago'].values

            for w, name in zip(ROLLING_WINDOWS_HOURS, ROLLING_WINDOW_NAMES):
                start = max(0, pos - (w - 1))
                window = lag1_history[start:pos + 1]
                test_df.at[next_idx, f'rolling_{name}_avg'] = np.mean(window)
                test_df.at[next_idx, f'rolling_{name}_std'] = (
                    np.std(window, ddof=1) if len(window) > 1 else 0.0
                )

            # Rebuild hourly momentum
            test_df.at[next_idx, 'sales_momentum_hourly'] = (
                test_df.at[next_idx, 'sales_1h_ago'] -
                test_df.at[next_idx, 'sales_same_hour_yesterday']
            )

            # Rebuild expanding mean
            if pos > 0:
                test_df.at[next_idx, 'expanding_mean'] = np.mean(lag1_history[:pos + 1])

            # Rebuild ratio
            rolling_1d = test_df.at[next_idx, 'rolling_1d_avg']
            test_df.at[next_idx, 'ratio_1h_vs_1d_avg'] = (
                test_df.at[next_idx, 'sales_1h_ago'] / (rolling_1d + 1e-8)
            )

print("\nRecursive hourly forecast complete for all products.")

Running Recursive Hourly Forecast for ALL Products...
  Processing hour 1/270: 2025-11-01 08:00:00
  Processing hour 46/270: 2025-11-06 08:00:00
  Processing hour 91/270: 2025-11-11 08:00:00
  Processing hour 136/270: 2025-11-16 08:00:00
  Processing hour 181/270: 2025-11-21 08:00:00
  Processing hour 226/270: 2025-11-26 08:00:00

Recursive hourly forecast complete for all products.


In [8]:
# ==========================================
# 8. ROLL UP TO DAILY & CALCULATE METRICS
# ==========================================
print("Rolling up hourly predictions to daily totals...")

test_df['Date_Only'] = test_df['Date'].dt.date

# Daily rollup per product
daily_rollup = test_df.groupby(['Product_Name', 'Date_Only']).agg({
    'Sales': 'sum',
    'XGB_Prediction': 'sum',
    'original_sales_1h_ago': 'sum'
}).reset_index()

# Build naive baseline from daily totals (sales_1_day_ago per product)
daily_rollup = daily_rollup.sort_values(['Product_Name', 'Date_Only']).reset_index(drop=True)
daily_rollup['sales_1_day_ago'] = daily_rollup.groupby(
    'Product_Name', observed=False)['Sales'].shift(1)
daily_rollup['sales_1_day_ago'] = daily_rollup['sales_1_day_ago'].fillna(
    daily_rollup['original_sales_1h_ago']
)


# ==========================================
# NORTH STAR METRIC FUNCTION
# ==========================================
# I'm using a tiered metric system here so different stakeholders get the right view:
#   - WAPE:  Best global metric — tells the business "what % of total inventory did we get wrong?"
#            Unlike MAPE, WAPE weights by volume so high-sellers matter more than low-sellers.
#   - MASE:  Best "is the ML working?" metric — if MASE < 1.0, my model beats naive lag-1 baseline.
#            If MASE > 1.0, I'd be better off just using yesterday's sales as my forecast.
#   - MAE:   Best ground-level metric — kitchen staff care about units, not percentages.
#   - Bias:  Directional error — negative means I'm under-predicting (running out of stock),
#            positive means I'm over-predicting (wasting food).

def calculate_north_star_metrics(evaluation_slice):
    """Calculate my North Star metrics for a given time horizon slice.

    Returns a dict with WAPE, MASE, MAE, Bias (and legacy metrics for backwards compatibility).
    These are the metrics I actually care about for decision-making:
      - WAPE for stakeholder reporting (volume-weighted accuracy)
      - MASE for model validation (am I beating a naive baseline?)
      - MAE + Bias for kitchen operations (units off, direction of error)
    """
    if len(evaluation_slice) == 0:
        return {m: 0 for m in ['WAPE', 'MASE', 'MAE', 'Bias', 'MSE', 'RMSE', 'MAPE', 'SMAPE']}

    actual_sales = evaluation_slice['Sales'].values.astype(float)
    predicted_sales = evaluation_slice['XGB_Prediction'].values.astype(float)
    naive_baseline = evaluation_slice['sales_1_day_ago'].values.astype(float)

    # --- NORTH STAR 1: WAPE (Weighted Absolute Percentage Error) ---
    # Sum of absolute errors / Sum of actual sales
    # This tells me the total % of inventory I got wrong across all products
    total_absolute_errors = np.sum(np.abs(actual_sales - predicted_sales))
    total_actual_sales = np.sum(np.abs(actual_sales))
    wape = total_absolute_errors / total_actual_sales if total_actual_sales > 0 else np.nan

    # --- NORTH STAR 2: MASE (Mean Absolute Scaled Error) ---
    # My model's MAE divided by naive baseline MAE
    # < 1.0 means my model adds value, > 1.0 means I should just use yesterday's number
    mae = mean_absolute_error(actual_sales, predicted_sales)
    naive_baseline_mae = mean_absolute_error(actual_sales, naive_baseline)
    mase = mae / naive_baseline_mae if naive_baseline_mae > 0 else np.nan

    # --- NORTH STAR 3: MAE (Mean Absolute Error in units) ---
    # The kitchen needs to know: "how many units off will I typically be?"
    # (mae already calculated above)

    # --- NORTH STAR 4: Bias (directional error) ---
    # Negative = I'm under-predicting (we'll run out of stock)
    # Positive = I'm over-predicting (we'll waste food)
    forecast_bias = np.mean(predicted_sales - actual_sales)

    # --- Legacy metrics (keeping for backwards compatibility) ---
    mse = mean_squared_error(actual_sales, predicted_sales)
    rmse = np.sqrt(mse)

    # MAPE — only on non-zero actuals to avoid division by zero
    nonzero_mask = actual_sales > 0
    mape = mean_absolute_percentage_error(actual_sales[nonzero_mask], predicted_sales[nonzero_mask]) if nonzero_mask.sum() > 0 else np.nan

    # SMAPE — symmetric version, less sensitive to zeros
    abs_errors = np.abs(predicted_sales - actual_sales)
    abs_denominator = (np.abs(actual_sales) + np.abs(predicted_sales)) / 2.0
    valid_mask = abs_denominator > 0
    smape = np.mean(abs_errors[valid_mask] / abs_denominator[valid_mask]) if valid_mask.sum() > 0 else np.nan

    return {'WAPE': wape, 'MASE': mase, 'MAE': mae, 'Bias': forecast_bias,
            'MSE': mse, 'RMSE': rmse, 'MAPE': mape, 'SMAPE': smape}


    y_true = df_slice['Sales'].values.astype(float)
    y_pred = df_slice['XGB_Prediction'].values.astype(float)
    y_naive = df_slice['sales_1_day_ago'].values.astype(float)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    # MAPE — guard against zero sales
    nonzero_mask = y_true > 0
    mape = mean_absolute_percentage_error(y_true[nonzero_mask], y_pred[nonzero_mask]) if nonzero_mask.sum() > 0 else np.nan

    # SMAPE
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    valid = denominator > 0
    smape = np.mean(numerator[valid] / denominator[valid]) if valid.sum() > 0 else np.nan

    # MASE
    naive_mae = mean_absolute_error(y_true, y_naive)
    mase = mae / naive_mae if naive_mae > 0 else np.nan

    forecast_bias = np.mean(y_pred - y_true)

    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'MAPE': mape,
            'SMAPE': smape, 'MASE': mase, 'Bias': forecast_bias}


# ==========================================
# 9. PER-PRODUCT DETAILED SCORECARDS
# ==========================================
target_date_1 = pd.to_datetime('2025-11-02').date()
target_date_7 = pd.to_datetime('2025-11-08').date()
target_date_30 = pd.to_datetime('2025-11-30').date()

# --- Detailed report for each product (like Bacon Bap in the daily model) ---
for product in all_products:
    p_daily = daily_rollup[daily_rollup['Product_Name'] == product].copy()
    p_daily = p_daily.sort_values('Date_Only').reset_index(drop=True)

    df_1_day = p_daily[p_daily['Date_Only'] == target_date_1]
    df_1_week = p_daily[(p_daily['Date_Only'] >= target_date_1) & (p_daily['Date_Only'] <= target_date_7)]
    df_1_month = p_daily[(p_daily['Date_Only'] >= target_date_1) & (p_daily['Date_Only'] <= target_date_30)]

    comparison_df = pd.DataFrame({
        'Metric': ['WAPE', 'MASE', 'MAE', 'Bias', 'MSE', 'RMSE', 'MAPE', 'SMAPE'],
        '1-Day (Nov 2)': list(calculate_north_star_metrics(df_1_day).values()),
        '1-Week (Nov 2-8)': list(calculate_north_star_metrics(df_1_week).values()),
        '1-Month (Nov 2-30)': list(calculate_north_star_metrics(df_1_month).values())
    })

    for col in comparison_df.columns[1:]:
        comparison_df[col] = comparison_df[col].apply(
            lambda x: f"{float(x):.4f}" if not np.isnan(x) else "N/A"
        )

    print(f"\n======================================================")
    print(f"  HOURLY XGBOOST METRICS (RECURSIVE, ROLLED UP): {product}")
    print(f"======================================================")
    print(comparison_df.to_string(index=False))

    # Reality check
    p_daily['XGB_Prediction_Rounded'] = p_daily['XGB_Prediction'].round().astype(int)
    p_daily['Mistake_Amount'] = (p_daily['Sales'] - p_daily['XGB_Prediction_Rounded']).abs()

    print(f"\n  REALITY CHECK: {product}")
    print(f"  --------------------------------------------------")
    print(p_daily[['Date_Only', 'Sales', 'XGB_Prediction_Rounded', 'Mistake_Amount']].head(10).to_string(index=False))

Rolling up hourly predictions to daily totals...

  HOURLY XGBOOST METRICS (RECURSIVE, ROLLED UP): Almd & Aprct Bar
Metric 1-Day (Nov 2) 1-Week (Nov 2-8) 1-Month (Nov 2-30)
  WAPE           N/A              N/A                N/A
  MASE           N/A              N/A                N/A
   MAE        0.0000           0.0000             0.0000
  Bias        0.0000           0.0000             0.0000
   MSE        0.0000           0.0000             0.0000
  RMSE        0.0000           0.0000             0.0000
  MAPE           N/A              N/A                N/A
 SMAPE           N/A              N/A                N/A

  REALITY CHECK: Almd & Aprct Bar
  --------------------------------------------------
 Date_Only  Sales  XGB_Prediction_Rounded  Mistake_Amount
2025-11-01      0                       0               0
2025-11-02      0                       0               0
2025-11-03      0                       0               0
2025-11-04      0                       0          

In [9]:
# ==========================================
# PER-PRODUCT NORTH STAR LEADERBOARD (sorted by Bias for kitchen staff)
# ==========================================
# I'm sorting by Bias here because that's what the kitchen actually needs to act on:
# - Products with large negative bias = we're consistently under-prepping (running out!)
# - Products with large positive bias = we're over-prepping (wasting food!)
# Then MAE tells them how many units they'll typically be off on any given day.

print("\n==================================================")
print("  PER-PRODUCT NORTH STAR METRICS")
print("  Sorted by Bias (most under-predicted first)")
print("==================================================")

target_date_1 = pd.to_datetime('2025-11-02').date()
target_date_30 = pd.to_datetime('2025-11-30').date()
per_product_metrics = []
for product_name in all_products:
    product_data = daily_rollup[daily_rollup['Product_Name'] == product_name].copy()
    product_month_data = product_data[(product_data['Date_Only'] >= target_date_1) & (product_data['Date_Only'] <= target_date_30)]
    product_metrics = calculate_north_star_metrics(product_month_data)
    product_metrics['Product'] = product_name
    per_product_metrics.append(product_metrics)

product_leaderboard_df = pd.DataFrame(per_product_metrics)
product_leaderboard_df = product_leaderboard_df[['Product', 'WAPE', 'MASE', 'MAE', 'Bias', 'RMSE']]

# Sort by Bias so the kitchen can see which items they're under/over-prepping most
product_leaderboard_df = product_leaderboard_df.sort_values('Bias')

for col in ['WAPE', 'MASE', 'MAE', 'Bias', 'RMSE']:
    product_leaderboard_df[col] = product_leaderboard_df[col].apply(
        lambda x: f"{x:.4f}" if not np.isnan(x) else "N/A"
    )

print(product_leaderboard_df.to_string(index=False))

# Quick interpretation guide
print("\n  INTERPRETATION GUIDE:")
print("  - WAPE < 0.20 = Good (within 20% of total volume)")
print("  - MASE < 1.00 = Model beats naive baseline (yesterday\'s sales)")
print("  - Bias < 0    = Under-predicting (stock-out risk)")
print("  - Bias > 0    = Over-predicting (waste risk)")

# ==========================================
# OVERALL AGGREGATE NORTH STAR METRICS
# ==========================================
# This is the "executive summary" — the single set of numbers I'd show to stakeholders.
# WAPE is the star here: it tells them what % of our total inventory forecast was wrong.

all_month = daily_rollup[(daily_rollup['Date_Only'] >= target_date_1) & (daily_rollup['Date_Only'] <= target_date_30)].copy()
overall_north_star = calculate_north_star_metrics(all_month)

print("\n==================================================")
print("  NORTH STAR METRICS: XGBoost Improved Hourly (All Products)")
print("==================================================")
print(f"  WAPE:  {overall_north_star['WAPE']:.4f}  <- % of total inventory wrong")
print(f"  MASE:  {overall_north_star['MASE']:.4f}  <- {'BEATING' if overall_north_star['MASE'] < 1 else 'LOSING TO'} naive baseline")
print(f"  MAE:   {overall_north_star['MAE']:.4f}  <- avg units off per product per day")
print(f"  Bias:  {overall_north_star['Bias']:.4f}  <- {'under-predicting' if overall_north_star['Bias'] < 0 else 'over-predicting'}")
print("  ---")
for metric_name in ['RMSE', 'MAPE', 'SMAPE']:
    val = overall_north_star[metric_name]
    print(f"  {metric_name:>6s}: {val:.4f}" if not np.isnan(val) else f"  {metric_name:>6s}: N/A")


  PER-PRODUCT NORTH STAR METRICS
  Sorted by Bias (most under-predicted first)
              Product   WAPE   MASE    MAE    Bias    RMSE
               Bakery 0.1479 0.8686 9.3153 -4.3393 11.6339
      P.Eggs on Toast 0.4807 1.1799 2.6852 -2.4905  3.0973
          Baked Beans 0.5296 0.8905 2.7943 -2.4249  3.3990
          Sausage Bap 0.2859 0.8226 3.4604 -1.8191  4.0112
     Avo, Egg & Bacon 0.6246 1.1043 2.1323 -1.7043  2.7025
Multiseed Toast Bread 0.6666 0.7179 1.9309 -1.6941  2.7250
              Tuna JP 0.6701 0.8935 1.8486 -1.6220  2.4686
     Breakfast Muffin 0.4514 0.7050 1.7745 -1.3132  2.2718
        Festive Stack 0.8276 1.4529 2.2545 -1.2176  2.8559
      S.Eggs on Toast 0.5695 0.8005 1.9047 -0.9916  2.3207
      Avo, Feta & Tom 0.6328 0.7693 1.3528 -0.9911  1.9188
    Brown Toast Bread 0.2393 0.7420 2.0213 -0.9684  2.5864
     Veggie Breakfast 0.6438 0.6438 1.4430 -0.9332  1.9973
          Poached Egg 0.5878 0.6831 1.7430 -0.7957  2.4608
              Egg Bap 0.7425 0.6497

In [10]:
# ==========================================
# 11. FEATURE IMPORTANCE
# ==========================================
print("\n==================================================")
print("  TOP 20 FEATURE IMPORTANCES (gain)")
print("==================================================")
importance = model.get_booster().get_score(importance_type='gain')
importance_sorted = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:20]
for feat, score in importance_sorted:
    print(f"  {feat:<35s} {score:.1f}")


  TOP 20 FEATURE IMPORTANCES (gain)
  sales_same_hour_last_week           1989.3
  sales_same_hour_2weeks_ago          1289.1
  rolling_7d_avg                      433.3
  Product_Name                        422.3
  sales_same_hour_yesterday           315.2
  hour_sin                            245.4
  expanding_mean                      243.0
  rolling_7d_std                      233.8
  is_afternoon                        216.8
  rolling_1d_avg                      199.7
  is_lunch_rush                       176.2
  hour_cos                            152.3
  is_morning_rush                     132.0
  rolling_3d_avg                      119.2
  sales_1h_ago                        114.7
  Is_Weekend                          108.6
  day_sin                             79.9
  day_cos                             50.3
  holiday_importance                  48.9
  ratio_1h_vs_1d_avg                  46.1


In [11]:
# ==========================================
# SAVE RESULTS — SQLite Storage Framework
# ==========================================
# I'm transitioning from flat CSVs to a proper two-table relational schema.
# This lets me track model performance over time and compare models in Tableau/PowerBI.
#
# Table 1: predictions_log — every single prediction (for actual vs predicted charts)
# Table 2: metrics_summary — aggregated scorecard (for "which model is winning?" queries)

import sqlite3
from datetime import datetime

os.makedirs('../results', exist_ok=True)

# --- Generate a unique run ID so I can track this specific model run ---
prediction_timestamp = datetime.now()
run_id = prediction_timestamp.strftime('%Y%m%d_%H%M') + '_XGBoost_Improved_Hourly'

print(f"Saving results for run: {run_id}")

# --- 1. Still save CSVs for quick access (backwards compatibility) ---
product_leaderboard_df.to_csv('../results/xgboost_improved_hourly_per_product_results.csv', index=False)

daily_rollup[['Product_Name', 'Date_Only', 'Sales', 'XGB_Prediction']].to_csv(
    '../results/xgboost_improved_hourly_daily_rollup.csv', index=False)
print("  CSVs saved to ../results/")

# --- 2. SQLite Storage Framework ---
# Connect to (or create) my model tracking database
database_path = '../results/model_tracking.db'
db_connection = sqlite3.connect(database_path)

with db_connection:
    # Create tables if they don't exist yet
    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS predictions_log (
            run_id TEXT,
            model_type TEXT,
            target_date TEXT,
            prediction_made_on TEXT,
            product_name TEXT,
            actual_sales REAL,
            predicted_sales REAL
        )
    """)

    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS metrics_summary (
            run_id TEXT,
            model_type TEXT,
            product_name TEXT,
            evaluation_horizon TEXT,
            WAPE REAL,
            MASE REAL,
            MAE REAL,
            Bias REAL
        )
    """)

    # --- Populate predictions_log ---
    predictions_for_db = daily_rollup[['Date_Only', 'Product_Name', 'Sales', 'XGB_Prediction']].copy()
    predictions_for_db = predictions_for_db.rename(columns={
        'Date_Only': 'target_date', 'Product_Name': 'product_name',
        'Sales': 'actual_sales', 'XGB_Prediction': 'predicted_sales'
    })
    predictions_for_db['run_id'] = run_id
    predictions_for_db['model_type'] = 'XGBoost_Improved_Hourly'
    predictions_for_db['prediction_made_on'] = str(prediction_timestamp)
    predictions_for_db['target_date'] = predictions_for_db['target_date'].astype(str)
    predictions_for_db.to_sql('predictions_log', db_connection, if_exists='append', index=False)
    print(f"  Logged {len(predictions_for_db)} predictions to predictions_log")

    # --- Populate metrics_summary (the scorecard) ---
    # This is where I can instantly query which model is "winning" across all runs

    evaluation_start = pd.to_datetime('2025-11-02').date()
    evaluation_week_end = pd.to_datetime('2025-11-08').date()
    evaluation_month_end = pd.to_datetime('2025-11-30').date()
    product_col_name = 'Product_Name'
    horizon_products = all_products

    month_slice = daily_rollup[(daily_rollup['Date_Only'] >= evaluation_start) & (daily_rollup['Date_Only'] <= evaluation_month_end)].copy()
    week_slice = month_slice[month_slice['Date_Only'] <= evaluation_week_end].copy()
    day_slice = month_slice[month_slice['Date_Only'] == evaluation_start].copy()
    horizons = {'1-Day': day_slice, '1-Week': week_slice, '1-Month': month_slice}

    metrics_rows = []

    # Per-product metrics at each horizon
    for product_name in horizon_products:
        for horizon_label, horizon_df in horizons.items():
            product_horizon_data = horizon_df[horizon_df[product_col_name] == product_name] if len(horizon_df) > 0 else horizon_df.iloc[0:0]
            if len(product_horizon_data) > 0:
                m_dict = calculate_north_star_metrics(product_horizon_data)
                metrics_rows.append({
                    'run_id': run_id,
                    'model_type': 'XGBoost_Improved_Hourly',
                    'product_name': product_name,
                    'evaluation_horizon': horizon_label,
                    'WAPE': m_dict.get('WAPE', np.nan),
                    'MASE': m_dict.get('MASE', np.nan),
                    'MAE': m_dict.get('MAE', np.nan),
                    'Bias': m_dict.get('Bias', np.nan),
                })

    # ALL_PRODUCTS aggregate row for each horizon
    for horizon_label, horizon_df in horizons.items():
        if len(horizon_df) > 0:
            m_dict = calculate_north_star_metrics(horizon_df)
            metrics_rows.append({
                'run_id': run_id,
                'model_type': 'XGBoost_Improved_Hourly',
                'product_name': 'ALL_PRODUCTS',
                'evaluation_horizon': horizon_label,
                'WAPE': m_dict.get('WAPE', np.nan),
                'MASE': m_dict.get('MASE', np.nan),
                'MAE': m_dict.get('MAE', np.nan),
                'Bias': m_dict.get('Bias', np.nan),
            })

    if metrics_rows:
        metrics_summary_df = pd.DataFrame(metrics_rows)
        metrics_summary_df.to_sql('metrics_summary', db_connection, if_exists='append', index=False)
        print(f"  Logged {len(metrics_summary_df)} metric rows to metrics_summary")

db_connection.close()
print(f"  SQLite database saved to {database_path}")
print(f"  Run ID: {run_id}")
print("\nDone! To query results later:")
print("  SELECT * FROM metrics_summary WHERE model_type=\'XGBoost_Improved_Hourly\' ORDER BY WAPE")
print("  SELECT * FROM predictions_log WHERE run_id=\'<run_id>\' AND product_name=\'Bacon Bap\'")

Saving results for run: 20260419_1550_XGBoost_Improved_Hourly
  CSVs saved to ../results/
  Logged 6660 predictions to predictions_log
  Logged 669 metric rows to metrics_summary
  SQLite database saved to ../results/model_tracking.db
  Run ID: 20260419_1550_XGBoost_Improved_Hourly

Done! To query results later:
  SELECT * FROM metrics_summary WHERE model_type='XGBoost_Improved_Hourly' ORDER BY WAPE
  SELECT * FROM predictions_log WHERE run_id='<run_id>' AND product_name='Bacon Bap'
